In [2]:
import pandas as pd

# -----------------------------
# 1. Public Parquet URL
# -----------------------------
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet"

# -----------------------------
# 2. Read Parquet file
# -----------------------------
try:
    df = pd.read_parquet(url)
    print("✅ Connection successful — file loaded.")
except Exception as e:
    print("❌ Error loading file:", e)
    raise

# -----------------------------
# 3. Basic validation
# -----------------------------
print("\n📌 First 5 rows:")
print(df.head())

print("\n📌 Shape (rows, columns):")
print(df.shape)

print("\n📌 Column names:")
print(df.columns.tolist())

✅ Connection successful — file loaded.

📌 First 5 rows:
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2019-01-01 00:46:40   2019-01-01 00:53:20              1.0   
1         1  2019-01-01 00:59:47   2019-01-01 01:18:59              1.0   
2         2  2018-12-21 13:48:30   2018-12-21 13:52:40              3.0   
3         2  2018-11-28 15:52:25   2018-11-28 15:55:45              5.0   
4         2  2018-11-28 15:56:57   2018-11-28 15:58:33              5.0   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0            1.5         1.0                  N           151           239   
1            2.6         1.0                  N           239           246   
2            0.0         1.0                  N           236           236   
3            0.0         1.0                  N           193           193   
4            0.0         2.0                  N           193           193   

   payment_type  f

In [3]:
import pandas as pd

# ----------------------------------------
# STEP 2 — DATA CLEANING & PREPROCESSING
# ----------------------------------------

# 1) Select relevant columns
relevant_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount"
]

df = df[relevant_columns]
print("Columns after selection:", df.columns.tolist())

# 2) Handle missing and invalid data

# Drop rows with missing critical values
df = df.dropna(subset=[
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "fare_amount"
])

# Remove duplicates
df = df.drop_duplicates()

# Filter invalid values
df = df[df["trip_distance"] > 0]
df = df[df["fare_amount"] > 0]

print("Rows after cleaning:", df.shape[0])

# 3) Standardise data types

# Convert datetime columns
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

# Convert numeric columns
numeric_cols = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows where conversion created NaN
df = df.dropna(subset=numeric_cols)

print("Data types after standardisation:")
print(df.dtypes)

Columns after selection: ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'fare_amount', 'tip_amount', 'total_amount']
Rows after cleaning: 7634764
Data types after standardisation:
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
fare_amount                     float64
tip_amount                      float64
total_amount                    float64
dtype: object


In [4]:
from sqlalchemy import create_engine

# ----------------------------------------
# SQL Server connection (Windows Auth)
# ----------------------------------------

engine = create_engine(
    "mssql+pyodbc://@BOLLAMHARSHIL\\SQLEXPRESS/SQL_Bhargavi_pappuri"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

# Test connection
try:
    with engine.connect() as conn:
        print("✅ Connected successfully to SQL Server")
except Exception as e:
    print("❌ Connection failed:", e)

✅ Connected successfully to SQL Server


In [13]:
df_10k = df.head(10000)
print(df_10k.shape)

(10000, 7)


In [14]:
from sqlalchemy import create_engine

engine = create_engine(
    "mssql+pyodbc://@BOLLAMHARSHIL\\SQLEXPRESS/SQL_Bhargavi_pappuri"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

df_10k.to_sql(
    name='yellow_taxi_2019',
    con=engine,
    if_exists='append',
    index=False
)

print("Inserted first 10,000 rows successfully")

Inserted first 10,000 rows successfully


In [ ]:
<img width="1907" height="1022" alt="SQLscreenshot" src="https://github.com/user-attachments/assets/12658f13-9d91-4c33-8b3a-20f261cd0a24" />